# SeqDiffuSeq EN→ZH — vast.ai Training Notebook

**Before running:** Sync code and data from your Mac:
```bash
./scripts/push_vastai.sh                     # code + tokenizer + BART weights
./scripts/push_vastai.sh --ckpts             # also sync existing checkpoints (to resume)
```

Checkpoints are saved to `VOLUME_DIR/ckpts/en-zh/` (persistent volume).

In [ ]:
import os, subprocess, glob, re, shutil

# ── CONFIG ───────────────────────────────────────────────────────────────
REPO_DIR    = "/root/SeqDiffuSeq"
DATA_SRC    = "/root/train_dataset"          # rsync'd from Mac
# ─────────────────────────────────────────────────────────────────────────

CKPT_DIR     = os.path.join(REPO_DIR, "ckpts", "en-zh")
DST_DATA_DIR = os.path.join(REPO_DIR, "data", "en-zh")
OUT_DIR      = os.path.join(CKPT_DIR, "inference_out")
PRETRAINED   = os.path.join(REPO_DIR, "pretrained", "bart-base")
VENV_PYTHON  = "python3"

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DST_DATA_DIR, exist_ok=True)
os.makedirs(PRETRAINED, exist_ok=True)

print(f"Repo      : {REPO_DIR}")
print(f"Checkpts  : {CKPT_DIR}")

## Phase 0 — Checkpoint Setup (skip if starting fresh)

Upload `model005000.pt` and `alpha_cumprod_step_*.npy` to the instance (e.g. via `scp` or the vast.ai file browser), then set `CKPT_STAGE` below and run the cell.

In [ ]:
# Directory where you uploaded the checkpoint files (set to "" to skip)
CKPT_STAGE = "/root"

if CKPT_STAGE:
    import glob as _glob
    staged = (
        [f for f in _glob.glob(os.path.join(CKPT_STAGE, "model*.pt"))
         if "ema" not in os.path.basename(f)]
        + _glob.glob(os.path.join(CKPT_STAGE, "ema_0.9999_*.pt"))
        + _glob.glob(os.path.join(CKPT_STAGE, "alpha_cumprod_step_*.npy"))
    )
    if not staged:
        print(f"No checkpoint files found in {CKPT_STAGE} — nothing to copy.")
    for src in staged:
        dst = os.path.join(CKPT_DIR, os.path.basename(src))
        if os.path.exists(dst):
            print(f"  Exists  {os.path.basename(src)}")
        else:
            shutil.copy2(src, dst)
            size_mb = os.path.getsize(dst) / 1024**2
            print(f"  Copied  {os.path.basename(src)}  ({size_mb:.0f} MB) → {CKPT_DIR}")
else:
    print("CKPT_STAGE is empty — skipping checkpoint copy.")

In [ ]:
# GPU Check
import torch
print("CUDA     :", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU      :", torch.cuda.get_device_name(0))
    print("VRAM     :", round(props.total_memory / 1024**3, 1), "GB")
    print("PyTorch  :", torch.__version__)

In [ ]:
# Install dependencies (vast.ai has CUDA + python3, no venv needed)
subprocess.run([
    "pip", "install", "-q",
    "bert-score", "blobfile", "datasets",
    "huggingface-hub==0.4.0",
    "mpi4py", "nltk", "numpy", "pandas", "protobuf",
    "rouge-score", "sacrebleu", "sacremoses",
    "scikit-learn", "scipy", "spacy",
    "tokenizers", "torchmetrics", "tqdm",
    "transformers==4.18.0",
], check=True)
print("Dependencies installed.")

## Phase 2 — Data

In [ ]:
# Verify source files exist in DATA_SRC (rsync'd from Mac)
required = [
    "train_clean.en", "train_clean.zh",
    "valid_clean.en", "valid_clean.zh",
    "test_clean.en",  "test_clean.zh",
]
all_ok = True
for fname in required:
    path = os.path.join(DATA_SRC, fname)
    if os.path.exists(path):
        n = sum(1 for _ in open(path, encoding="utf-8"))
        print(f"  ✓  {fname:30s}  {n:>8,} lines")
    else:
        print(f"  ✗  {fname:30s}  NOT FOUND")
        all_ok = False
if not all_ok:
    raise FileNotFoundError("Run push_vastai.sh to sync train_dataset/ first.")

In [ ]:
# Copy cleaned files to repo data dir (original names, no changes needed in training args)
file_mapping = {
    "train_clean.en": "train.en", "train_clean.zh": "train.zh",
    "valid_clean.en": "valid.en", "valid_clean.zh": "valid.zh",
    "test_clean.en":  "test.en",  "test_clean.zh":  "test.zh",
}
for src_name, dst_name in file_mapping.items():
    src = os.path.join(DATA_SRC, src_name)
    dst = os.path.join(DST_DATA_DIR, dst_name)
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f"  Copied  {src_name} → {dst_name}")
    else:
        print(f"  Exists  {dst_name}")

## Phase 3 — Tokenizer

Tokenizer was already trained in Session 02 (`vocab.json` + `merges.txt` in `data/en-zh/`).
This cell just verifies they were synced — **no retraining needed**.

In [ ]:
tok_vocab  = os.path.join(DST_DATA_DIR, "vocab.json")
tok_merges = os.path.join(DST_DATA_DIR, "merges.txt")
if os.path.exists(tok_vocab) and os.path.exists(tok_merges):
    print("Tokenizer ready:", tok_vocab)
else:
    raise FileNotFoundError(
        f"vocab.json / merges.txt missing in {DST_DATA_DIR}\n"
        "Run push_vastai.sh to sync SeqDiffuSeq/data/en-zh/ first."
    )

## Phase 4 — BART-base Weights

In [ ]:
# Download bart-base if not already synced (safe_serialization=False for pytorch_model.bin)
if os.path.exists(os.path.join(PRETRAINED, "pytorch_model.bin")):
    print("bart-base already cached →", PRETRAINED)
else:
    from transformers import BartConfig, BartTokenizerFast, BartModel
    print("Downloading facebook/bart-base…")
    BartConfig.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED)
    BartTokenizerFast.from_pretrained("facebook/bart-base").save_pretrained(PRETRAINED)
    BartModel.from_pretrained("facebook/bart-base").save_pretrained(
        PRETRAINED, safe_serialization=False
    )
    print("Saved to", PRETRAINED)

## Phase 5 — Training

Auto-resumes from the latest checkpoint in `CKPT_DIR` if one exists.

In [ ]:
os.makedirs(os.path.join(CKPT_DIR, "log"), exist_ok=True)

# Auto-resume: find the latest non-EMA checkpoint (model######.pt)
model_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "model*.pt")))
model_ckpts = [c for c in model_ckpts if "ema" not in os.path.basename(c)]
resume_ckpt = model_ckpts[-1] if model_ckpts else ""
if resume_ckpt:
    step_match = re.search(r"model0*(\d+)\.pt", os.path.basename(resume_ckpt))
    step_num = int(step_match.group(1)) if step_match else "?"
    print(f"Resuming from step {step_num}: {os.path.basename(resume_ckpt)}")
else:
    print("No checkpoint found — starting from scratch.")

args = [
    VENV_PYTHON, "-u", "main.py",
    "--checkpoint_path",    CKPT_DIR,
    "--src",                "en",
    "--tgt",                "zh",
    "--train_txt_path",     "./data/en-zh/train",
    "--val_txt_path",       "./data/en-zh/valid",
    "--dataset",            "en-zh",
    "--config_name",        PRETRAINED,
    "--diffusion_steps",    "1000",
    "--noise_schedule",     "sqrt",
    "--sequence_len",       "64",
    "--sequence_len_src",   "128",
    "--batch_size",         "64",     # 3090/4090 has ~24 GB VRAM; bump from 16
    "--lr",                 "1e-4",
    "--lr_anneal_steps",    "100000",
    "--resume_checkpoint",  resume_ckpt,
    "--warmup",             "500",
    "--save_interval",      "5000",
    "--eval_interval",      "2500",
    "--log_interval",       "100",
    "--schedule_update_stride", "200",
    "--loss_update_granu",  "20",
    "--encoder_layers",     "6",
    "--decoder_layers",     "6",
    "--num_heads",          "12",
    "--in_channel",         "768",
    "--out_channel",        "768",
    "--num_channels",       "3072",
    "--vocab_size",         "32005",
    "--dropout",            "0.3",
    "--predict_xstart",     "True",
    "--seed",               "42",
    "--schedule_sampler",   "uniform",
    "--init_pretrained",    "True",
    "--freeze_embeddings",  "False",
    "--use_pretrained_embeddings", "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"]  = "0"
env["DIFFUSION_BLOB_LOGDIR"] = os.path.join(CKPT_DIR, "log")
env["TRANSFORMERS_OFFLINE"]  = "1"

print("Starting training… logs →", os.path.join(CKPT_DIR, "log"))
result = subprocess.run(args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    print("Exit code:", result.returncode)

## Phase 6 — Inference

In [ ]:
# Find latest EMA checkpoint and time schedule
ema_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, "ema_0.9999_*.pt")))
if not ema_ckpts:
    raise FileNotFoundError(f"No EMA checkpoint in {CKPT_DIR}")
model_path = ema_ckpts[-1]
print(f"Checkpoint : {os.path.basename(model_path)}")

schedule_files = sorted(glob.glob(os.path.join(CKPT_DIR, "alpha_cumprod_step_*.npy")))
if not schedule_files:
    raise FileNotFoundError(f"No alpha_cumprod_step_*.npy in {CKPT_DIR}")
schedule_path = schedule_files[-1]
print(f"Schedule   : {os.path.basename(schedule_path)}")

test_src  = os.path.join(REPO_DIR, "data", "en-zh", "test.en")
num_test  = sum(1 for _ in open(test_src, encoding="utf-8"))
print(f"Test lines : {num_test:,}  (num_samples=-1 → full set)")

inf_args = [
    VENV_PYTHON, "-u", "inference_main.py",
    "--model_name_or_path", model_path,
    "--val_txt_path",       "./data/en-zh/test",
    "--out_dir",            CKPT_DIR,
    "--time_schedule_path", schedule_path,
    "--diffusion_steps",    "200",
    "--num_samples",        "-1",
    "--batch_size",         "100",
    "--sequence_len",       "64",
    "--sequence_len_src",   "128",
    "--top_p",              "-1",
    "--clamp",              "no_clamp",
    "--use_ddim",           "True",
    "--seed",               "42",
    "--generate_by_q",      "False",
    "--generate_by_mix",    "False",
]

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["TRANSFORMERS_OFFLINE"] = "1"

print(f"\nStarting inference on {num_test:,} sentences…")
result = subprocess.run(inf_args, cwd=REPO_DIR, env=env, capture_output=True, text=True)
print(result.stdout[-3000:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
    print("Exit code:", result.returncode)

## Phase 7 — BLEU Evaluation

In [ ]:
decoded_files = sorted([
    f for f in glob.glob(os.path.join(CKPT_DIR, "ema_*.pt.samples_*.txt"))
    if "raw-output-ids" not in f
])

if not decoded_files:
    print("No decoded output file found in:", CKPT_DIR)
else:
    output_file = decoded_files[-1]
    m = re.search(r"ema_[\d.]+_(\d+)", os.path.basename(output_file))
    short = f"eval_step{m.group(1)}" if m else "eval"
    os.makedirs(OUT_DIR, exist_ok=True)
    eval_csv     = os.path.join(OUT_DIR, short + ".csv")
    eval_summary = os.path.join(OUT_DIR, short + "_summary.txt")
    print(f"Evaluating: {output_file}\n")

    eval_script = f"""
import json, csv, sacrebleu

with open({repr(output_file)}, "r", encoding="utf-8") as f:
    pairs = [json.loads(l.strip()) for l in f if l.strip()]

hypotheses = [p[0] for p in pairs]
references  = [p[1] for p in pairs]

src_path = {repr(os.path.join(REPO_DIR, "data", "en-zh", "test.en"))}
with open(src_path, "r", encoding="utf-8") as f:
    sources = [l.strip() for l in f if l.strip()]

n = min(len(sources), len(hypotheses))
sources, hypotheses, references = sources[:n], hypotheses[:n], references[:n]

bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='char')
bleu_13a  = sacrebleu.corpus_bleu(hypotheses, [references], tokenize='13a')

with open({repr(eval_csv)}, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["source_en", "hypothesis_zh", "reference_zh"])
    for src, hyp, ref in zip(sources, hypotheses, references):
        writer.writerow([src, hyp, ref])

summary = "\\n".join([
    f"Output file     : {repr(output_file)}",
    f"Num samples     : {{n}}",
    "",
    f"SacreBLEU (char): {{bleu_char.score:.2f}}",
    f"SacreBLEU (13a) : {{bleu_13a.score:.2f}}",
])
print(summary)
with open({repr(eval_summary)}, "w", encoding="utf-8") as f:
    f.write(summary)
print(f"\\nCSV     : {repr(eval_csv)}")
print(f"Summary : {repr(eval_summary)}")
"""

    result = subprocess.run(
        [VENV_PYTHON, "-c", eval_script],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])